# BGE-M3 ColBERT late-interaction pilot

1,200 official queries × up to 50 existing candidate docs × 2 verbatim passage windows. This is a **full shortlist scoring run, gated by pilot**, not a BTC submission or a relevance evaluation. No qrels are present. Exact model revision and input SHA are checked. One T4 GPU is intentional; resume from atomic checkpoint. No private keys are printed.


In [ ]:
%pip install -q "FlagEmbedding==1.3.5" "transformers==4.51.3" "accelerate==1.6.0" "sentence-transformers==3.4.1" "huggingface-hub==0.30.2" "pyarrow>=18,<21" "pandas>=2.2,<3"

In [ ]:
import gc, hashlib, json, os, time, zipfile
from pathlib import Path
import numpy as np, pandas as pd, torch
from FlagEmbedding import BGEM3FlagModel

ROOT=Path('/kaggle/input')
inputs=list(ROOT.rglob('colbert_full_pairs.parquet'))
manifests=list(ROOT.rglob('manifest.json'))
if len(inputs)!=1: raise RuntimeError(f'Expected one pilot Parquet, got {len(inputs)}')
manifest_path=inputs[0].with_name('manifest.json')
if not manifest_path.is_file(): raise RuntimeError('Pilot manifest missing beside Parquet')
manifest=json.loads(manifest_path.read_text())
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for block in iter(lambda:f.read(1<<20),b''): h.update(block)
    return h.hexdigest()
if manifest['schema']!='colbert_full_v1' or sha(inputs[0])!=manifest['pairs_sha256']:
    raise RuntimeError('Input fingerprint mismatch')
pairs=pd.read_parquet(inputs[0]); assert len(pairs)==manifest['pairs']
WORK=Path('/kaggle/working/colbert_full');WORK.mkdir(exist_ok=True)
checkpoint=WORK/'scores.parquet'
done=pd.read_parquet(checkpoint) if checkpoint.exists() else pd.DataFrame()
done_ids=set(done.pair_id) if len(done) else set()
print({'pairs':len(pairs),'remaining':len(pairs)-len(done_ids),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},flush=True)
if not torch.cuda.is_available(): raise RuntimeError('GPU required; stop without CPU fallback')

In [ ]:
t0=time.time()
MODEL='BAAI/bge-m3'
REVISION=manifest['model_revision']
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    token=None
if token:
    from huggingface_hub import login
    login(token=token,add_to_git_credential=False)
from huggingface_hub import snapshot_download
model_path=snapshot_download(repo_id=MODEL,revision=REVISION,token=token)
model=BGEM3FlagModel(model_path,use_fp16=True,devices=['cuda:0'],batch_size=8,
                    query_max_length=256,passage_max_length=384)
print({'model_load_seconds':round(time.time()-t0,1),'gpu_allocated_gb':round(torch.cuda.memory_allocated()/1e9,2)},flush=True)

In [ ]:
# Query embeddings are computed once. Passage vectors are never stored long-term.
queries=pairs[['query_id','query_text']].drop_duplicates('query_id').set_index('query_id').query_text.to_dict()
qvec={}
for qid,text in queries.items():
    out=model.encode([text],batch_size=1,max_length=256,
                     return_dense=False,return_sparse=False,return_colbert_vecs=True)
    qvec[int(qid)]=out['colbert_vecs'][0]
print({'queries_encoded':len(qvec)},flush=True)

In [ ]:
# Atomic checkpoints every 100 pairs. Resume is keyed by pair_id and input hash.
if checkpoint.exists():
    state=json.loads((WORK/'state.json').read_text())
    if state['pairs_sha256']!=manifest['pairs_sha256'] or state['revision']!=REVISION:
        raise RuntimeError('Checkpoint belongs to different input/model')
records=done.to_dict('records') if len(done) else []
pending=pairs.loc[~pairs.pair_id.isin(done_ids)]
start=time.time();n_initial=len(records)
for b in range(0,len(pending),8):
    batch=pending.iloc[b:b+8]
    output=model.encode(batch.passage_text.astype(str).tolist(),batch_size=8,max_length=384,
                        return_dense=False,return_sparse=False,return_colbert_vecs=True)
    for row,pvec in zip(batch.itertuples(index=False),output['colbert_vecs']):
        score=float(model.colbert_score(qvec[int(row.query_id)],pvec))
        if not np.isfinite(score): raise RuntimeError('Non-finite ColBERT score')
        records.append({'pair_id':row.pair_id,'query_id':int(row.query_id),'doc_id':int(row.doc_id),
                        'doc_pick_rank':int(row.doc_pick_rank),'passage_rank':int(row.passage_rank),
                        'colbert_score':score})
    if (b+len(batch))%1000<8 or b+len(batch)==len(pending):
        stage=WORK/'scores.building.parquet'
        pd.DataFrame(records).to_parquet(stage,index=False)
        os.replace(stage,checkpoint)
        (WORK/'state.json').write_text(json.dumps({'pairs_sha256':manifest['pairs_sha256'],'revision':REVISION,
            'complete':len(records),'elapsed_seconds':round(time.time()-start,1)}))
        elapsed=max(time.time()-start,.001); speed=(len(records)-n_initial)/elapsed
        print({'complete':len(records),'total':len(pairs),'pairs_per_second':round(speed,2),
               'eta_minutes':round((len(pairs)-len(records))/max(speed,.001)/60,1)},flush=True)
    del output
    gc.collect()
assert len(records)==len(pairs) and len(set(x['pair_id'] for x in records))==len(pairs)
report={'input_sha256':manifest['pairs_sha256'],'model':MODEL,'revision':REVISION,
        'pairs':len(records),'queries':len(qvec),'seconds_this_session':round(time.time()-start,1),
        'pairs_per_second_this_session':round((len(records)-n_initial)/max(time.time()-start,.001),2),
        'scores_sha256':sha(checkpoint),'status':'complete','btc_score':None}
(WORK/'run_manifest.json').write_text(json.dumps(report,indent=2))
archive=Path('/kaggle/working/colbert_full_outputs.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(checkpoint,'scores.parquet');z.write(WORK/'run_manifest.json','run_manifest.json')
print({'output':str(archive),'sha256':sha(archive),'report':report},flush=True)